# 🌾 AgroCrédito Colombia - Modelado y Evaluación

**Equipo:** Andrés, Sebastián, Julián, Yuri - Talento Tech 2

Este cuaderno entrena el modelo de Regresión Lineal para predecir tasas de crédito rural.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib
import os

plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 1. Carga de Datos

In [ ]:
df = pd.read_csv('data/creditos_rurales_limpio.csv')
print(f"Dataset: {df.shape[0]} filas, {df.shape[1]} columnas")
df.head()

## 2. Definición de Variables

In [ ]:
features = [
    'Edad', 'Experiencia_Anios', 'Hectareas', 'Ingresos_Mensuales_COP',
    'Monto_Prestamo_COP', 'Plazo_Meses', 'Garantia_Respaldada',
    'Subsidio_Gobierno', 'Tecnologias_Usadas', 'Tiene_Energia_Solar',
    'Tiene_Riego'
]

target = 'Tasa_Interes_EA'

X = df[features].copy()
y = df[target].copy()

print(f"Variables predictoras: {len(features)}")
print(f"Variable objetivo: {target}")
print(f"\nFeatures: {features}")

## 3. División Train/Test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Entrenamiento: {len(X_train)} registros")
print(f"Prueba: {len(X_test)} registros")

## 4. Entrenamiento del Modelo

In [ ]:
modelo = LinearRegression()
modelo.fit(X_train, y_train)

print("Modelo entrenado exitosamente")
print(f"\nIntercepto: {modelo.intercept_:.6f}")
print(f"\nCoeficientes:")
for feat, coef in zip(features, modelo.coef_):
    print(f"  {feat}: {coef:.6f}")

## 5. Predicciones

In [ ]:
y_pred = modelo.predict(X_test)

# Comparar predicciones vs valores reales
comparison = pd.DataFrame({
    'Real': y_test.values[:10],
    'Predicho': y_pred[:10],
    'Error': abs(y_test.values[:10] - y_pred[:10])
})
print("Primeras 10 predicciones:")
comparison

## 6. Métricas de Evaluación

In [ ]:
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("=" * 50)
print("MÉTRICAS DEL MODELO")
print("=" * 50)
print(f"R² Score:  {r2:.4f} ({r2*100:.2f}%)")
print(f"MAE:       {mae:.4f} puntos porcentuales")
print(f"RMSE:      {rmse:.4f}")
print("=" * 50)

## 7. Análisis de Residuos

In [ ]:
residuos = y_test - y_pred

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Distribución de residuos
axes[0].hist(residuos, bins=30, color='green', edgecolor='black')
axes[0].set_title('Distribución de Residuos')
axes[0].set_xlabel('Residuo')
axes[0].axvline(x=0, color='red', linestyle='--')

# Predicción vs Real
axes[1].scatter(y_test, y_pred, alpha=0.5, color='green')
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[1].set_title('Predicción vs Real')
axes[1].set_xlabel('Valor Real')
axes[1].set_ylabel('Predicción')

# Residuos vs Predicción
axes[2].scatter(y_pred, residuos, alpha=0.5, color='green')
axes[2].axhline(y=0, color='red', linestyle='--')
axes[2].set_title('Residuos vs Predicción')
axes[2].set_xlabel('Predicción')
axes[2].set_ylabel('Residuo')

plt.tight_layout()
plt.show()

## 8. Importancia de Variables

In [ ]:
coef_df = pd.DataFrame({
    'Variable': features,
    'Coeficiente': modelo.coef_
}).sort_values('Coeficiente')

plt.figure(figsize=(10, 6))
plt.barh(coef_df['Variable'], coef_df['Coeficiente'], color='viridis')
plt.title('Coeficientes del Modelo (Impacto en Tasa)', fontsize=14)
plt.xlabel('Coeficiente')
plt.tight_layout()
plt.show()

print("\nImpacto de cada variable:")
print("=" * 50)
for _, row in coef_df.iterrows():
    impacto = "reduce" if row['Coeficiente'] < 0 else "aumenta"
    print(f"  {row['Variable']}: {impacto} {abs(row['Coeficiente']):.4f}% por unidad")

## 9. Guardar Modelo

In [ ]:
model_data = {
    'modelo': modelo,
    'caracteristicas': features,
    'coeficientes': dict(zip(features, modelo.coef_)),
    'intercepto': modelo.intercept_,
    'metricas': {'r2': r2, 'mae': mae, 'rmse': rmse}
}

joblib.dump(model_data, 'modelo_creditos.pkl')
print("Modelo guardado: modelo_creditos.pkl")

## Resumen del Modelado

| Métrica | Valor |
|---------|-------|
| R² Score | 89.2% |
| MAE | 0.75 pp |
| RMSE | 0.96 |

### Variables con Mayor Impacto:
1. **Subsidio LEC Finagro:** -4.34% E.A.
2. **Garantía FAG:** -2.36% E.A.
3. **Energía Solar:** -0.69% E.A.
4. **Cada tecnología:** -0.35% E.A.
5. **Experiencia:** -0.04% E.A. por año